# [RSO-90]: Tip/Tilt variations using double-zernikes during filter changes
Author: Bruno Quint
Date: 2026-01-30

We have seen gradients in PSF sigma across the whole field of view of LSSTCam.  
Given that the field of view is large, this is not surprising.  
However, as part of the investigation of the focus shift during filter offset,  
we started to wonder how large are these gradients and if their directions change after a filter change.  
One way of doing that would be by gathering the PSF Sigma on each detector and  
fitting a plane with the gradient and its direction.  
However, the measured PSF Sigma is affected by the atmosphere.  
Instead, we will use here the tip and tilt components from the double-Zernikes polynomials,  
which better isolate the optical state of the system.  

[RSO-90]: https://rubinobs.atlassian.net/browse/RSO-57

## Notebook Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
day_obs = 20260128
repo = "/repo/embargo"

# day_obs = 20251108
# repo = "/repo/main"

collection = "LSSTCam/runs/nightlyValidation"
dataset = "aggregateAOSVisitTableAvg"

# CORNER_DETECTORS = (191, 192, 195, 196, 199, 200, 203, 204)
CORNER_DETECTORS = [191, 195, 199, 203]

In [13]:
from astropy.time import Time
from bokeh.embed import file_html
from bokeh.io import export_png, output_notebook
from bokeh.plotting import show

import lsst.sitcom.tn175 as tn175

# Make sure Bokeh initializes properly
output_notebook()

Loading BokehJS ...

## Night Summary

Before going to double zernikes, let me confirm that we are seeing the same thing as other plots show in other places. 

### Query data from ConsDB

Query ConsDB to generate a table for the filter focus offset study, aggregating and transforming relevant exposure and detector data.

This function performs the following operations:
  - Queries ConsDB for exposures matching the specified observation day and image types ('science' or 'acq'), joining relevant tables to collect:
    - Sequence number, day of observation, physical rotator angle, telescope elevation, airmass, observation start/end times, focus position, observation reason, physical filter, band, PSF sigma median, AOS FWHM, CCD PSF sigma, CCD Z4, CCD Z11, and detector ID.
  - Filters out rows with airmass equal to zero.
  - Parses observation start and end times into datetime objects.
  - Converts PSF sigma values to FWHM at zenith and 500nm for both visit-level and CCD-level measurements.
  - Removes the original PSF sigma columns after conversion.
  - Constructs a unique visit identifier and sets it as the DataFrame index.
  - For each detector in `corner_detectors`, creates columns for Zernike coefficients (z4 and z11) specific to that detector, containing values only for matching detector rows.
  - Aggregates the table by visit, keeping the first value for most columns, the mean for CCD Zernike coefficients (z4, z11), and the first value for each per-detector Zernike column.
  - The final table contains one row per visit (for visits with corner detectors), with columns:
    - visit, seq, day_obs, physical_rotator_angle, altitude, airmass,
      obs_start, obs_end, focus_z, observation_reason, band_p, band,
      aos_fwhm, fwhm_zenith_500nm_median, ccd_fwhm_zenith_500nm,
      ccd_z4_mean, ccd_z11_mean,
      and for each corner detector: ccd_z4_d{det}, ccd_z11_d{det}.

In [4]:
# Try to load cached data
cdb_table_filename = f"consdb_aggregated_table_{day_obs}.csv"
cdb_table_path = os.path.join("./data", cdb_table_filename) 

if os.path.exists(cdb_table_path):
    print(f"Load existing file: {cdb_table_path}")
    cdb_table = pd.read_csv(cdb_table_path)
    print(f"  Successfully loaded file: {cdb_table_path}")


# If there is no cached data, query and save it
else:
    print(f"Could not find file: {cdb_table_path}")
    print(f"Start a ConsDB Client")
    # Required to use ConsDb, following documentation above
    os.environ["no_proxy"] += ",.consdb"

    # Initialize ConsDb
    cdb_client = ConsDbClient("http://consdb-pq.consdb:8080/consdb")    
    print(f"Successfully loaded a ConsDB Client")

    # Perform the query and aggregation
    print(f"Query ConsDB for data from day_obs={day_obs}")
    cdb_table = tn175.query.table_for_filter_focus_offset_study(
        cdb_client, day_obs, corner_detectors=CORNER_DETECTORS)
    print(f"Data queried successfully")

    # Save for future interactions
    print(f"Save file: {cdb_table_path}")
    os.makedirs("./data", exist_ok=True)
    cdb_table.to_csv(cdb_table_path)
    print(f"File saved successfully.")

Load existing file: ./data/consdb_aggregated_table_20260128.csv
  Successfully loaded file: ./data/consdb_aggregated_table_20260128.csv


### Query datasets using Butler for Double Zernikes

In [10]:
# Try to load cached data
dz_table_filename = f"dz_aggregated_table_{day_obs}.csv"
dz_table_path = os.path.join("./data", dz_table_filename) 

if os.path.exists(dz_table_path):
    print(f"Load existing file: {dz_table_path}")
    dz_table = pd.read_csv(dz_table_path)
    print(f"  Successfully loaded file: {dz_table_path}")


# If there is no cached data, query save it
else:
    print(f"Query Butler for Zernike data from day_obs={day_obs}")
    dz_table = tn175.query.butler_for_double_zernike_calculation(
        repo, collection, dataset, day_obs, CORNER_DETECTORS)
    print(f"Data queried successfully")

    # Save for future interactions
    print(f"Save file: {dz_table_path}")
    os.makedirs("./data", exist_ok=True)
    dz_table.to_csv(dz_table_path)
    print(f"File saved successfully.")

Query Butler for Zernike data from day_obs=20260128
Got a list with 340 dataset results.
Adding Zernike coefficients to the dataframe...
Processing 340 visits and 4 corner detectors.


Processing:   0%|          | 0/1360 [00:00<?, ?it/s]

  Failed: visit 2026012800005, detector 199: index 0 is out of bounds for axis 0 with size 0
Finished adding Zernike coefficients.
DataFrame shape after adding Zernike coefficients: (340, 97)
Computing double Zernike coefficients for each visit...


  0%|          | 0/340 [00:00<?, ?it/s]

Finished computing double Zernike coefficients.
Data queried successfully
Save file: ./data/dz_aggregated_table_20260128.csv
File saved successfully.


## Put everything together

In [11]:
full_table = tn175.utils.merge_double_zernikes(cdb_table, dz_table)

In [31]:
mask = full_table["observation_reason"] == "filter_offsets_test"

my_plot = tn175.plot.plot_night_timeline_combined(full_table[mask], day_obs=day_obs)
show(my_plot)

Now that we have our plots, let's save the outputs:

In [32]:
from bokeh.io import output_file, save

# Setup the outputs
base_path = "./plots"
base_name = f"rso90_filter_offset_study_{day_obs}"

# Make sure you have a place to store the plots
os.makedirs(base_path, exist_ok=True)

# Save the plots as HTML
output_file(os.path.join(base_path, f"{base_name}.html"))
save(my_plot)

'/home/b/bquint/notebooks/lsst-sitcom/sitcomtn-175/notebooks/plots/rso90_filter_offset_study_20260128.html'

If you want a PNG file, you will have to click on the download button.   
Unfortunately, Bokeh requires a browser simulator that is not available in lsst-stack.